# Phase 1 sweep — 2×2 transfer study

Runs `{mlp, dueling} × {vanilla, double}` in scratch and transfer conditions on
CartPole-v1 → LunarLander-v3.

**Colab sessions time out.** Everything here is built around that: runs checkpoint
every 25 episodes into Google Drive, and re-running a cell resumes rather than
restarts. If the session dies, reconnect and re-run the same cell.

Run the stages in order — `transfer` needs `source` to have finished, because each
transfer run loads the CartPole checkpoint from its own cell and seed.

## 1. Setup

In [ ]:
!pip install -q 'gymnasium[box2d]' swig
import tensorflow as tf, gymnasium as gym
print('TF', tf.__version__, '| gym', gym.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'none (CPU is fine — the nets are tiny)')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Persist results outside the ephemeral VM so a timeout costs nothing.
OUT_ROOT = '/content/drive/MyDrive/rl-transfer/runs'
REPO = '/content/Transfer-Learning-in-Double-and-Dueling-DQNs'

import os
os.makedirs(OUT_ROOT, exist_ok=True)
print('results ->', OUT_ROOT)

In [ ]:
# Point this at your fork/branch, or upload the repo to Drive and copy it in.
REPO_URL = 'https://github.com/<owner>/Transfer-Learning-in-Double-and-Dueling-DQNs.git'

import os
if not os.path.exists(REPO):
    !git clone -q $REPO_URL $REPO
%cd $REPO
!git log --oneline -1

## 2. Check the plan before spending compute

`--dry-run` lists what would run and what is already complete.

In [ ]:
!python experiments/sweep.py --seeds 0-9 --stage all --out-root $OUT_ROOT --dry-run

## 3. Pilot first — 2 seeds

Validates the pipeline and gives a real per-run timing before committing to the
full sweep. Check that CartPole actually learns (reward climbing well above ~30)
before going further: in the published runs the DDQN source never learned its
source task, and that went unnoticed.

In [ ]:
!python experiments/sweep.py --seeds 0 1 --stage source --out-root $OUT_ROOT

In [ ]:
# Sanity gate: did the source task actually get solved?
!python experiments/aggregate.py --out-root $OUT_ROOT

## 4. Full sweep, stage by stage

Re-run any cell after a timeout; completed runs are skipped and partial runs resume.

In [ ]:
!python experiments/sweep.py --seeds 0-9 --stage source   --out-root $OUT_ROOT

In [ ]:
!python experiments/sweep.py --seeds 0-9 --stage baseline --out-root $OUT_ROOT

In [ ]:
!python experiments/sweep.py --seeds 0-9 --stage transfer --out-root $OUT_ROOT

## 5. Results

`aggregate.py` applies the final-100-episode evaluation window identified in
Phase 0; `stats.py` reports the within-cell transfer effect, which is the
comparison that separates transferability from target-task suitability.

In [ ]:
!python experiments/aggregate.py --out-root $OUT_ROOT

In [ ]:
!python experiments/stats.py --per-seed $OUT_ROOT/per_seed.csv

## 6. Keep the results

`per_seed.csv` is the artifact every number in the paper should come from. Copy it
back into the repo and commit it alongside the manuscript revision.

In [ ]:
import pandas as pd
df = pd.read_csv(f'{OUT_ROOT}/per_seed.csv')
print(df.groupby(['env_id', 'arm']).size())
df.head()